# Audio → Zero-Contamination Diarization Experiment

This notebook mirrors the SonicStudio **Experiment** funnel for vocal diarization
and zero-contamination speaker curation. Separation (Demucs / RoFormer) is performed
in upstream notebooks (e.g. `pipeline1.ipynb`) or batch pipelines, feeding clean speech stems
directly into this multi-stage gate funnel.

| Gate | Stage | Engine / Model | Purpose |
|---|---|---|---|
| **1** | Primary Diarization | Sortformer / DiariZen / Pyannote | Fast initial speaker turn hypothesis |
| **2** | Hungarian Consensus | Secondary Diarizer | Eliminates unconfirmed ghost turns |
| **3a** | Context Collar | Distance-aware Erosion | Dynamic boundary erosion at speaker handoffs |
| **3b** | Syllable Lock | PhoWhisper / MMS / Whisper | Snaps turn edges to nearest word/syllable boundary |
| **3c** | Energy Valley Snapping | Acoustic RMS Valley Detection | Snaps cuts into micro-energy minima (< -30 dB) |
| **3d** | Smart Segmentation | ASR Punctuation + Valley Cuts | Slices long turns into optimal TTS sentences (3–10s) |
| **4** | Homogeneity Filter | WeSpeaker ResNet34 (cosine sim) | Prunes intra-turn speaker intrusion / contamination |
| **5a** | Direct-Audio Verifier | Gemini 3.8 / Gemma 4 Audio | Purity + word completeness (không bị lẹm chữ) |
| **5b** | Speaker Count Gate | VibeVoice-ASR Int8 | Acoustic multi-speaker overlap rejection |


In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import os
import sys
from dataclasses import replace
from html import escape as html_escape
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio as IPythonAudio, HTML, clear_output, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

# Works when Jupyter starts in src/notebooks/ as documented, and also from repo subdirectories.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Could not find the repository root (pyproject.toml).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.data_paths import DATA_DIR
from src.utils.AudioClass import Audio
from src.utils.AudioCutter import AudioCutter
from src.yt_crawler.YtCrawlerClass import YtCrawler

# Source Separation Backends

# Speaker Diarization Backends & Schema
from src.diarization import (
    BaseDiarizer,
    ClusteringWorkerDiarizer,
    DiariZenDiarizer,
    DiariZenWorkerDiarizer,
    DiarizationFilter,
    DiarizationModelInfo,
    DiarizationResult,
    clean_speaker_turns,
    PyannoteDiarizer,
    SortformerDiarizer,
    SortformerWorkerDiarizer,
    Speaker,
    SpeakerTurn,
    SpeakerVerifier,
    ThreeDSpeakerWorkerDiarizer,
    ZeroContaminationConfig,
    ZeroContaminationResult,
    DEFAULT_EMBEDDING_MODEL_ID,
)

# Pipeline Stages & Defaults
from src.diarization.zero_contamination import (
    DEFAULT_COLLAR_EROSION_S,
    DEFAULT_COMPETITOR_ONSET,
    DEFAULT_ENERGY_FRAME_LEN_MS,
    DEFAULT_ENERGY_HOP_LEN_MS,
    DEFAULT_ENERGY_SEARCH_WINDOW_S,
    DEFAULT_ENERGY_VALLEY_FLOOR_DB,
    DEFAULT_HANDOFF_RISK_DISTANCE_S,
    DEFAULT_HOMOGENEITY_HOP_S,
    DEFAULT_HOMOGENEITY_WINDOW_S,
    DEFAULT_MIN_HOMOGENEITY_SIMILARITY,
    DEFAULT_MIN_SPLIT_PAUSE_S,
    DEFAULT_MIN_TURN_DURATION_S,
    DEFAULT_SILENCE_TAIL_BUFFER_S,
    DEFAULT_TARGET_MAX_DURATION_S,
    DEFAULT_TARGET_MIN_DURATION_S,
    DEFAULT_TARGET_OFFSET,
    DEFAULT_TARGET_ONSET,
    DEFAULT_TRANSITION_EXCLUSION_S,
    align_and_lock_syllable_boundaries,
    apply_context_aware_collar,
    compute_consensus_turns,
    filter_by_embedding_homogeneity,
    filter_by_foundation_models,
    run_zero_contamination_pipeline,
    smart_segment_speaker_turns,
    snap_boundaries_to_acoustic_valleys,
)


## Input Audio & Device Allocation

Specify your input audio target and compute device. Each downstream stage has its own local parameters directly inside its cell for maximum interactive flexibility.

In [ ]:
# Input audio: point to a pre-separated speech/vocal audio file or YouTube URL
URL = "https://www.youtube.com/watch?v=REPLACE_ME"
LOCAL_AUDIO_PATH: Path | None = Path(
    "../../.data/mel_roformer/out/Khám_phá_Top_Đại_Học_Siêu_Hot_Bách_Khoa,_Khoa_Học_Tự_Nhiên,_Công_Nghệ_VyUni_Ep.2_khối_KH-KT-CN__mEHOvkGFz54.wav"
)  # e.g. Path("/path/to/vocal_stem.wav")

# Compute devices & credentials
DEVICE = "cuda:0"
HF_TOKEN = os.getenv("HF_TOKEN")

# Stage attrition tracking & interactive inspection
stage_stats = {}
current_speech: DiarizationResult | None = None


def as_result(turns: list[SpeakerTurn], stage_name: str = "stage") -> DiarizationResult:
    """Wrap intermediate turns into a canonical DiarizationResult for interactive inspection."""
    speaker_ids = sorted({turn.speaker_id for turn in turns})
    return DiarizationResult(
        schema_version="2.0",
        audio_id=speech_audio.source_id,
        speakers=[Speaker(speaker_id=s) for s in speaker_ids],
        turns=list(turns),
        source_audio=speech_audio,
        model=DiarizationModelInfo(
            backend="zero-contamination-notebook",
            model_id=f"stage_{stage_name}",
        ),
    )


def record_stage(name: str, turns) -> list[SpeakerTurn]:
    """Record turn attrition and update current_speech as a DiarizationResult."""
    global current_speech
    turns = list(turns)
    stage_stats[name] = {
        "turns": len(turns),
        "speech_duration_s": round(sum(turn.duration_s for turn in turns), 2),
    }
    current_speech = as_result(turns, name)
    display(stage_stats[name])
    return turns


## Model Initializers — Flexible Diarizer Selection

The factory below exposes explicit initialization for supported speaker diarizer backends.


In [ ]:
def init_diarizer(
    backend: str,
    *,
    device: str = DEVICE,
    token: str | None = None,
    onset: float = DEFAULT_TARGET_ONSET,
    offset: float = DEFAULT_TARGET_OFFSET,
    model_id: str | None = None,
    isolated: bool = True,
    **kwargs,
) -> BaseDiarizer:
    """Initialize any supported speaker diarizer backend for primary or secondary stages.

    Supported backends:
      - 'sortformer': SortformerWorkerDiarizer (isolated subprocess) or SortformerDiarizer
      - 'diarizen': DiariZenWorkerDiarizer (isolated subprocess) or DiariZenDiarizer
      - 'pyannote' / 'pyannote_community': PyannoteDiarizer (community-1)
      - 'pyannote_31': PyannoteDiarizer (v3.1)
      - 'clustering': ClusteringWorkerDiarizer
      - '3dspeaker': ThreeDSpeakerWorkerDiarizer
    """
    b = backend.lower().strip()
    tok = token or os.getenv("HF_TOKEN")
    if b in {"sortformer", "nemo-sortformer"}:
        if isolated:
            return SortformerWorkerDiarizer(device=device, token=tok, onset=onset, offset=offset, **kwargs)
        return SortformerDiarizer(device=device, token=tok, onset=onset, offset=offset, **kwargs)
    elif b in {"diarizen", "diarizen_large_s80_v2"}:
        if isolated:
            return DiariZenWorkerDiarizer(device=device, token=tok, **kwargs)
        return DiariZenDiarizer(device=device, token=tok, **kwargs)
    elif b in {"pyannote", "pyannote_community"}:
        mid = model_id or "pyannote/speaker-diarization-community-1"
        return PyannoteDiarizer(model_id=mid, device=device, token=tok, **kwargs)
    elif b in {"pyannote_31", "pyannote_3.1"}:
        mid = model_id or "pyannote/speaker-diarization-3.1"
        return PyannoteDiarizer(model_id=mid, device=device, token=tok, **kwargs)
    elif b in {"clustering", "clustering_worker"}:
        return ClusteringWorkerDiarizer(device=device, token=tok, **kwargs)
    elif b in {"3dspeaker", "3d_speaker"}:
        return ThreeDSpeakerWorkerDiarizer(device=device, token=tok, **kwargs)
    else:
        raise ValueError(
            f"Unsupported diarizer backend: {backend}. "
            f"Choose from 'sortformer', 'diarizen', 'pyannote', 'pyannote_31', 'clustering', '3dspeaker'."
        )


## Input — Load Pre-Separated Speech Audio

Point to a clean vocal stem (pre-separated in `pipeline1.ipynb` or saved under `.data/`)
or download directly from YouTube.


In [ ]:
if LOCAL_AUDIO_PATH is not None and Path(LOCAL_AUDIO_PATH).is_file():
    speech_audio = Audio.from_file(LOCAL_AUDIO_PATH)
    print(f"Loaded speech audio file: {speech_audio.path}")
else:
    crawler = YtCrawler(
        output_dir=DATA_DIR / "notebook" / "zero_contamination" / "downloads",
        work_dir=DATA_DIR / "notebook" / "zero_contamination" / "crawl_work",
    )
    speech_audio = crawler.download(URL)

source_audio = speech_audio  # alias for backwards compatibility

display(speech_audio.metadata())
speech_audio.notebook_display()


## Diarization Filter & Turn Cleanup Configuration (Global / Per-Step Controls)

Configurable filter criteria and turn cleanup (matching SonicStudio Diarization controls) usable at **any stage** of the pipeline:
- **`diar_filter`**: Pre-configured filter instance using the parameters below.
- **`apply_filter()`**: Helper to filter `current_speech` after any step: `apply_filter().display()`
- **Fluent API**: `current_speech.filter(speakers='SPEAKER_00', min_duration_s=1.0).display()`


In [ ]:
# Filter Criteria (mirrors SonicStudio Diarization Tab filters)
FILTER_TARGET_SPEAKER: str | None = None       # e.g., "SPEAKER_00" to isolate one speaker
EXCLUDE_SPEAKERS: list[str] | None = None       # e.g., ["SPEAKER_01"]
FILTER_MIN_DURATION_S: float | None = 1.0       # Exclude short turns (< 1.0s)
FILTER_MAX_DURATION_S: float | None = None      # Exclude turns longer than this (seconds)
EXCLUDE_OVERLAPPING_SPEECH: bool = True        # Exclude turns overlapping another speaker

# Turn Cleanup Settings (A-B-A jitter correction, collars, and same-speaker gap merging)
ENABLE_TURN_CLEANUP: bool = False              # Enable if you want to merge gaps/trim collars
CLEANUP_MIN_DURATION_S: float = 0.5
CLEANUP_MERGE_GAP_S: float = 1.0
CLEANUP_BOUNDARY_COLLAR_S: float = 0.04
CLEANUP_JITTER_MAX_DURATION_S: float = 3.0

diar_filter = DiarizationFilter(
    speakers=FILTER_TARGET_SPEAKER,
    exclude_speakers=EXCLUDE_SPEAKERS,
    min_duration_s=FILTER_MIN_DURATION_S,
    max_duration_s=FILTER_MAX_DURATION_S,
    exclude_overlap=EXCLUDE_OVERLAPPING_SPEECH,
    clean_turns=ENABLE_TURN_CLEANUP,
    min_turn_duration_s=CLEANUP_MIN_DURATION_S,
    merge_same_speaker_gap_s=CLEANUP_MERGE_GAP_S,
    boundary_collar_s=CLEANUP_BOUNDARY_COLLAR_S,
    jitter_max_duration_s=CLEANUP_JITTER_MAX_DURATION_S,
)


def apply_filter(result: DiarizationResult | None = None) -> DiarizationResult:
    """Filter current_speech or a given DiarizationResult using global diar_filter."""
    target = result if result is not None else current_speech
    if target is None:
        raise ValueError("No DiarizationResult available to filter yet.")
    return target.filter_with(diar_filter)


## Experiment Stage 1 — Primary Diarization

In [ ]:
# Stage 1 Parameters
PRIMARY_BACKEND = "sortformer"  # "sortformer", "diarizen", "pyannote", "pyannote_31"
TARGET_ONSET = 0.80
TARGET_OFFSET = 0.65

primary_diarizer = init_diarizer(
    backend=PRIMARY_BACKEND,
    device=DEVICE,
    token=HF_TOKEN,
    onset=TARGET_ONSET,
    offset=TARGET_OFFSET,
)

with primary_diarizer:
    primary_result: DiarizationResult = primary_diarizer.diarize(speech_audio)

current_turns = record_stage(
    "1_primary", sorted(primary_result.turns, key=lambda turn: turn.start_s)
)


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 2 — Dual-Engine Mutual Hungarian Consensus

In [ ]:
# Stage 2 Parameters
ENABLE_CONSENSUS = True
SECONDARY_BACKEND = "diarizen"  # "diarizen", "sortformer", "pyannote", "pyannote_31"
SECONDARY_DEVICE = DEVICE       # "cuda:0", "cuda:1", or "cpu"

if ENABLE_CONSENSUS:
    secondary_diarizer = init_diarizer(
        backend=SECONDARY_BACKEND,
        device=SECONDARY_DEVICE,
        token=HF_TOKEN,
    )
    with secondary_diarizer:
        secondary_result: DiarizationResult = secondary_diarizer.diarize(speech_audio)
    current_turns, speaker_mapping = compute_consensus_turns(
        current_turns, secondary_result.turns, speech_audio.duration_s
    )
    current_turns = record_stage("2_consensus", current_turns)
    display({"speaker_mapping": speaker_mapping})
else:
    print("Stage 2 (Consensus) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 3a — Context-Aware Collar & Handoff Guard

In [ ]:
# Stage 3a Parameters
ENABLE_CONTEXT_COLLAR = True
BOUNDARY_COLLAR_S = 0.35
HANDOFF_RISK_S = 0.80
SILENCE_TAIL_S = 0.027
MIN_TURN_DURATION_S = 0.80
TRANSITION_EXCLUSION_S = 0.50

if ENABLE_CONTEXT_COLLAR:
    current_turns, collar_audits = apply_context_aware_collar(
        current_turns,
        collar_s=BOUNDARY_COLLAR_S,
        handoff_risk_s=HANDOFF_RISK_S,
        silence_tail_s=SILENCE_TAIL_S,
        min_duration_s=MIN_TURN_DURATION_S,
        transition_exclusion_s=TRANSITION_EXCLUSION_S,
        audio_duration_s=speech_audio.duration_s,
    )
    current_turns = record_stage("3a_context_collar", current_turns)
else:
    print("Stage 3a (Context collar) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 3b — Syllable / Word Forced-Alignment Lock

In [ ]:
# Stage 3b Parameters
ENABLE_SYLLABLE_ALIGNMENT = True
ALIGNER_ENGINE = "whisper_timestamped"  # "whisper_timestamped", "mms_fa", "remote_whisper"
ALIGNER_MODEL = "vinai/PhoWhisper-small"  # "vinai/PhoWhisper-small", "vinai/PhoWhisper-large"
ALIGNER_LANGUAGE = "vi"
ALIGNER_DEVICE = "cuda:0"                  # "cpu" recommended to avoid VRAM contention
ALIGNER_ENDPOINT = os.getenv("WHISPER_ENDPOINT")

if ENABLE_SYLLABLE_ALIGNMENT:
    current_turns, alignment_audits = align_and_lock_syllable_boundaries(
        speech_audio,
        current_turns,
        aligner_engine=ALIGNER_ENGINE,
        aligner_model=ALIGNER_MODEL,
        aligner_language=ALIGNER_LANGUAGE,
        aligner_endpoint=ALIGNER_ENDPOINT,
        aligner_device=ALIGNER_DEVICE,
        token=HF_TOKEN,
    )
    current_turns = record_stage("3b_word_lock", current_turns)
else:
    print("Stage 3b (Syllable alignment) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 3c — Micro-Energy Valley Snapping

In [ ]:
# Stage 3c Parameters
ENABLE_ENERGY_SNAPPING = True
ENERGY_SEARCH_WINDOW_S = 0.15
ENERGY_VALLEY_FLOOR_DB = -30.0
ENERGY_FRAME_LEN_MS = 2.0
ENERGY_HOP_LEN_MS = 0.5

if ENABLE_ENERGY_SNAPPING:
    current_turns, energy_audits = snap_boundaries_to_acoustic_valleys(
        speech_audio,
        current_turns,
        search_window_s=ENERGY_SEARCH_WINDOW_S,
        energy_floor_db=ENERGY_VALLEY_FLOOR_DB,
        frame_len_ms=ENERGY_FRAME_LEN_MS,
        hop_len_ms=ENERGY_HOP_LEN_MS,
    )
    current_turns = record_stage("3c_energy_snap", current_turns)
else:
    print("Stage 3c (Energy snapping) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 3d — Intelligent ASR & Pause-Guided Turn Segmentation (TTS Sentence Sizing)

Segments long turns into optimal TTS training slices (3–10s) using ASR punctuation and breathing pauses, snapping cut points to local acoustic energy valleys.

In [ ]:
# Stage 3d Parameters
ENABLE_SMART_SEGMENTATION = True
TARGET_MAX_DURATION_S = 10.0
TARGET_MIN_DURATION_S = 3.0
MIN_SPLIT_PAUSE_S = 0.20

if ENABLE_SMART_SEGMENTATION:
    current_turns, segment_audits = smart_segment_speaker_turns(
        speech_audio,
        current_turns,
        max_duration_s=TARGET_MAX_DURATION_S,
        min_duration_s=TARGET_MIN_DURATION_S,
        min_pause_s=MIN_SPLIT_PAUSE_S,
        search_window_s=ENERGY_SEARCH_WINDOW_S,
        frame_len_ms=ENERGY_FRAME_LEN_MS,
        hop_len_ms=ENERGY_HOP_LEN_MS,
    )
    current_turns = record_stage("3d_smart_segmentation", current_turns)
else:
    print("Stage 3d (Smart segmentation) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 4 — WeSpeaker Sliding-Window Homogeneity

In [ ]:
# Stage 4 Parameters
ENABLE_HOMOGENEITY = True
HOMOGENEITY_DEVICE = DEVICE     # "cuda:0", "cuda:1", or "cpu"
HOMOGENEITY_WINDOW_S = 1.00
HOMOGENEITY_HOP_S = 0.25
MIN_HOMOGENEITY_SIMILARITY = 0.75

if ENABLE_HOMOGENEITY:
    current_turns, homogeneity_audits = filter_by_embedding_homogeneity(
        speech_audio,
        current_turns,
        window_s=HOMOGENEITY_WINDOW_S,
        hop_s=HOMOGENEITY_HOP_S,
        min_similarity=MIN_HOMOGENEITY_SIMILARITY,
        device=HOMOGENEITY_DEVICE,
        token=HF_TOKEN,
    )
    current_turns = record_stage("4_homogeneity", current_turns)
else:
    print("Stage 4 (Homogeneity) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 5a — Gemma/Gemini Direct-Audio Verifier

Verifies acoustic speaker purity and word completeness (không bị lẹm chữ).

In [ ]:
# Stage 5a Parameters
ENABLE_DIRECT_AUDIO = True
DIRECT_AUDIO_BACKEND = "gemini"         # "gemini" or "gemma4"
DIRECT_AUDIO_MODEL = "gemini-3.8-flash"  # "gemini-3.8-flash", "gemini-3.1-flash-lite", "unsloth/gemma-4-12b-it-GGUF"
DIRECT_AUDIO_CONCURRENCY = 10           # Parallel requests (Gemini: 10, Gemma 4: 1)
DIRECT_AUDIO_TIMEOUT_S = 120.0
DIRECT_AUDIO_MAX_TOKENS = 1024
DIRECT_AUDIO_ENDPOINT = os.getenv("UNSLOTH_ENDPOINT")  # only needed for local gemma4

if ENABLE_DIRECT_AUDIO:
    current_turns, direct_audio_audits = filter_by_foundation_models(
        speech_audio,
        current_turns,
        enable_gemma=True,
        enable_vibevoice=False,
        gemma_backend=DIRECT_AUDIO_BACKEND,
        gemma_model=DIRECT_AUDIO_MODEL,
        gemma_endpoint=DIRECT_AUDIO_ENDPOINT,
        gemma_concurrency=DIRECT_AUDIO_CONCURRENCY,
        gemma_timeout_s=DIRECT_AUDIO_TIMEOUT_S,
        gemma_max_output_tokens=DIRECT_AUDIO_MAX_TOKENS,
    )
    current_turns = record_stage("5a_direct_audio", current_turns)
else:
    print("Stage 5a (Direct audio verifier) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Experiment Stage 5b — VibeVoice-ASR Speaker-Count Verifier

In [ ]:
# Stage 5b Parameters
ENABLE_VIBEVOICE = True
VIBEVOICE_MODEL_ID = "Dubedo/VibeVoice-ASR-HF-INT8"
VIBEVOICE_DEVICE = "cuda:0"              # dedicated secondary GPU recommended
VIBEVOICE_ENDPOINT = os.getenv("VIBEVOICE_ENDPOINT")
MAX_SECONDARY_SPEECH_S = 0.0

if ENABLE_VIBEVOICE:
    current_turns, vibevoice_audits = filter_by_foundation_models(
        speech_audio,
        current_turns,
        enable_gemma=False,
        enable_vibevoice=True,
        vibevoice_model_id=VIBEVOICE_MODEL_ID,
        vibevoice_device=VIBEVOICE_DEVICE,
        vibevoice_endpoint=VIBEVOICE_ENDPOINT,
        max_secondary_speech_s=MAX_SECONDARY_SPEECH_S,
    )
    current_turns = record_stage("5b_vibevoice", current_turns)
else:
    print("Stage 5b (VibeVoice verifier) skipped.")


In [ ]:
current_speech.display()

# Or inspect this stage with the global filter applied:
# apply_filter().display()


## Assemble and Persist Canonical Diarization Result

In [ ]:
speaker_ids = sorted({turn.speaker_id for turn in current_turns})
consensus_tag = f"+{SECONDARY_BACKEND}" if ENABLE_CONSENSUS else ""
model_label = f"{PRIMARY_BACKEND}{consensus_tag}+notebook-gates"

diarization_result = DiarizationResult(
    schema_version="2.0",
    audio_id=speech_audio.source_id,
    speakers=[Speaker(speaker_id=speaker_id) for speaker_id in speaker_ids],
    turns=current_turns,
    source_audio=speech_audio,
    model=DiarizationModelInfo(
        backend="zero-contamination-notebook",
        model_id=model_label,
    ),
)
result_path = diarization_result.save(
    DATA_DIR / "notebook" / "zero_contamination" / "results"
)
print(f"Diarization result saved to: {result_path.resolve()}")
print(f"Result output folder:        {result_path.parent.resolve()}")
display({"saved_to": str(result_path.resolve()), "funnel": stage_stats})


## Apply Final Filtering & Post-Processing

Applies the global `diar_filter` defined at the top of the notebook to produce `filtered_diarization_result`.


In [ ]:
# Apply the global diar_filter to the final canonical result
filtered_diarization_result = diar_filter.apply(diarization_result)

print(f"Original turns: {diarization_result.turn_count} ({diarization_result.speaker_count} speakers, {diarization_result.total_speech_duration_s:.1f}s total)")
print(f"Filtered turns: {filtered_diarization_result.turn_count} ({filtered_diarization_result.speaker_count} speakers, {filtered_diarization_result.total_speech_duration_s:.1f}s total)")
display(filtered_diarization_result.to_dict()["summary"])


In [ ]:
# Export clean turn audio cuts to disk
from src.utils.AudioCutter import AudioCutter

cuts_dir = DATA_DIR / "notebook" / "zero_contamination" / "cuts" / speech_audio.source_id
cutter = AudioCutter(output_dir=cuts_dir)
cut_clips = cutter.cut_batch(
    speech_audio,
    [(t.start_s, t.end_s) for t in filtered_diarization_result.turns],
    show_progress=True,
    desc="Exporting clean audio cuts",
)

print(f"Exported {len(cut_clips)} clean audio segments to:")
print(f"  Audio cuts folder: {cuts_dir.resolve()}")


## Diarization Result Notebook Viewer & Plotting

Interactive inspection and plotting logic is built directly into the `DiarizationResult` class:
- `result.display()` (or `result.notebook_display()`): Launches the interactive widget viewer with speaker filtering, transcript search, raw/refined boundary comparison, waveform context, and lazy per-turn audio playback.
- `result.plot()`: Renders a static matplotlib timeline / Gantt chart of speaker turns and overlaps across the full audio duration.
- `result.plot_turn(index)`: Plots the context waveform around a specific turn highlighting raw vs refined boundary adjustments.


In [ ]:
# Launch the interactive DiarizationResult viewer directly from the class
filtered_diarization_result.display()

# Optional: render a static matplotlib timeline of speaker turns
# filtered_diarization_result.plot()

# Optional: inspect waveform context around turn #0
# if filtered_diarization_result.turns:
#     filtered_diarization_result.plot_turn(0)
